# 纳指 ETF 低溢价轮动策略（每周检查）

策略逻辑（4 支纳指 ETF）：

- 每周一检查一次（若周一不是交易日，则本周不检查）。
- 计算**上一个交易日**各 ETF 的 `discount_rate` 作为信号（避免未来函数）。SQLite: `adj_factor_etf.discount_rate`，单位为 **1%**（1.0 = 1 个百分点）。
- 找到当日溢价最低（`discount_rate` 最小）的 ETF。
- 如果它不是当前持仓，且相对当前持仓 **至少低 1.0**（即低 1 个百分点）=> 当日开盘全仓切换到该 ETF。

数据依赖：
- 价格：`etf_daily`（用于开盘交易 / 收盘估值）
- 折溢价率：`adj_factor_etf.discount_rate`

如本地数据库缺数据，先运行：`bash scripts/fetch_data.sh`（或 `python scripts/fetch_data.py`）。


In [ ]:
# 环境与依赖（不强依赖 pandas）
import _bootstrap  # noqa: F401 (adds src/ to sys.path)

import datetime as dt
import math

import numpy as np

from qs.backtester.runner import (
    load_calendar_bars_for_symbols_from_sqlite,
    run_backtest,
)
from qs.sqlite_utils import connect_sqlite
from qs.strategy.etf_min_premium_weekly import ETFMinPremiumWeeklyStrategy


In [ ]:
# 可选：绘图库（若导入失败则仅不画图）
try:
    import matplotlib
    import matplotlib.pyplot as plt

    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    matplotlib = None
    plt = None
    print("matplotlib unavailable, skip plotting:", repr(e))


In [ ]:
# 回测参数
DB_PATH = _bootstrap.RAW_DB_PATH

# 这 4 支纳指 ETF（本地库默认能解析到 .SZ/.SH；请按你的库实际情况调整）
SYMBOLS = [
    "159501.SZ",
    "513390.SH",
    "159696.SZ",
    "513100.SH",
]

END_DATE = None  # e.g. "20251231"
INIT_CASH = 1_000_000.0

# 交易/估值是否使用复权价（open/close * adj_factor / base_factor）
USE_ADJUSTED = True

# 每周一检查；若周一不是交易日，本周不检查
MONDAY_ONLY = True

# 切换阈值：新最低溢价 ETF 必须比当前持仓低至少 1.0（= 1 个百分点）
MIN_IMPROVEMENT = 1.0

if not DB_PATH.exists():
    raise FileNotFoundError(f"Missing DB: {DB_PATH} (run fetch/sync scripts first)")
print("DB:", DB_PATH)

# 自动选择回测起点：取 4 只 ETF 在 `etf_daily` 与 `discount_rate` 的共同覆盖起始日
con = connect_sqlite(DB_PATH, read_only=True)
try:
    cover_daily = {}
    cover_premium = {}
    cover_adj = {}
    for s in SYMBOLS:
        mn, mx = con.execute(
            "SELECT MIN(trade_date), MAX(trade_date) FROM etf_daily WHERE ts_code=?",
            [s],
        ).fetchone()
        cover_daily[s] = (str(mn) if mn else None, str(mx) if mx else None)
        pmn, pmx = con.execute(
            "SELECT MIN(trade_date), MAX(trade_date) FROM adj_factor_etf WHERE ts_code=? AND discount_rate IS NOT NULL",
            [s],
        ).fetchone()
        cover_premium[s] = (str(pmn) if pmn else None, str(pmx) if pmx else None)
        if USE_ADJUSTED:
            amn, amx = con.execute(
                "SELECT MIN(trade_date), MAX(trade_date) FROM adj_factor_etf WHERE ts_code=?",
                [s],
            ).fetchone()
            cover_adj[s] = (str(amn) if amn else None, str(amx) if amx else None)
finally:
    con.close()

missing_daily = [s for s, (mn, _mx) in cover_daily.items() if mn is None]
missing_premium = [s for s, (mn, _mx) in cover_premium.items() if mn is None]
if missing_daily:
    raise RuntimeError(f"Missing etf_daily data for: {missing_daily}")
if missing_premium:
    raise RuntimeError(f"Missing discount_rate data for: {missing_premium}")

START_DATE_DAILY = max(mn for mn, _mx in cover_daily.values() if mn is not None)
START_DATE_PREMIUM = max(mn for mn, _mx in cover_premium.values() if mn is not None)
START_DATE = max(START_DATE_DAILY, START_DATE_PREMIUM)

if USE_ADJUSTED:
    missing_adj = [s for s, (mn, _mx) in cover_adj.items() if mn is None]
    if missing_adj:
        print("WARN: adj_factor_etf missing for", missing_adj, "; fallback USE_ADJUSTED=False")
        USE_ADJUSTED = False
    else:
        START_DATE_ADJ = max(mn for mn, _mx in cover_adj.values() if mn is not None)
        START_DATE = max(START_DATE, START_DATE_ADJ)

print("Coverage:")
for s in SYMBOLS:
    print(" ", s, "daily", cover_daily[s][0], "->", cover_daily[s][1], "premium", cover_premium[s][0], "->", cover_premium[s][1])
print("USE_ADJUSTED:", USE_ADJUSTED)
print("START_DATE (overlap):", START_DATE)


In [ ]:
# 运行回测（qs 回测框架）
bars = load_calendar_bars_for_symbols_from_sqlite(
    db_path=DB_PATH,
    table="etf_daily",
    symbols=SYMBOLS,
    start_date=START_DATE,
    end_date=END_DATE,
)
print("calendar bars:", len(bars), "from", bars[0].trade_date, "to", bars[-1].trade_date)

strat = ETFMinPremiumWeeklyStrategy(
    db_path_raw=str(DB_PATH),
    symbols=SYMBOLS,
    start_date=START_DATE,
    end_date=END_DATE,
    use_adjusted=USE_ADJUSTED,
    monday_only=MONDAY_ONLY,
    min_improvement=MIN_IMPROVEMENT,
)

res = run_backtest(
    bars=bars,
    strategy=strat,
    initial_cash=INIT_CASH,
    symbol="",
    enable_trade_log=False,
    mark_error_policy="warn",
)

print(f"Final equity: {res.final_equity:,.2f}  Total return: {(res.final_equity/res.initial_cash-1):.2%}")
print(f"Max DD: {res.max_drawdown:.2%}  ({res.dd_peak} -> {res.dd_trough})")
print("Risk (daily):", dict(res.risk))
print("Weekly checks:", len(strat.get_check_history()), "Trades:", len(res.broker.trades))


In [ ]:
# 策略净值序列（list-based） + 单 ETF 买入持有对比（与 USE_ADJUSTED 对齐）
trade_dates = [p.trade_date for p in res.equity_curve]
equities = [float(p.equity) for p in res.equity_curve]
nav = [e / float(INIT_CASH) for e in equities]
dates = [dt.datetime.strptime(d, "%Y%m%d") for d in trade_dates]

print("points:", len(nav), "first:", trade_dates[0], "last:", trade_dates[-1])
print("nav last:", nav[-1])

# Buy&Hold 用 close 计算净值；若 USE_ADJUSTED=True，则用 close*adj_factor（归一到最新因子）以对齐策略。
base_adj: dict[str, float] = {}
if USE_ADJUSTED:
    con = connect_sqlite(DB_PATH, read_only=True)
    try:
        for s in SYMBOLS:
            row = con.execute(
                "SELECT adj_factor FROM adj_factor_etf WHERE ts_code=? ORDER BY trade_date DESC LIMIT 1",
                [s],
            ).fetchone()
            base_adj[s] = float(row[0]) if row and row[0] else 1.0
    finally:
        con.close()

def load_close_map(ts_code: str) -> dict[str, float]:
    con = connect_sqlite(DB_PATH, read_only=True)
    try:
        rows = con.execute(
            """
            SELECT d.trade_date, d.close, COALESCE(af.adj_factor, 1.0) AS adj_factor
            FROM etf_daily d
            LEFT JOIN adj_factor_etf af USING(ts_code, trade_date)
            WHERE d.ts_code=? AND d.trade_date>=?
            ORDER BY d.trade_date
            """,
            [ts_code, START_DATE],
        ).fetchall()
    finally:
        con.close()

    out: dict[str, float] = {}
    base = base_adj.get(ts_code) or 1.0
    if base == 0:
        base = 1.0
    for d, close, adj in rows:
        if close is None:
            continue
        px = float(close)
        if USE_ADJUSTED:
            px = px * float(adj) / float(base)
        out[str(d)] = px
    return out


def align_close(trade_dates: list[str], close_map: dict[str, float]) -> list[float] | None:
    out: list[float] = []
    last = None
    for d in trade_dates:
        if d in close_map:
            last = close_map[d]
        out.append(last if last is not None else float("nan"))
    if not out or all(math.isnan(x) for x in out):
        return None
    first = next((x for x in out if not math.isnan(x)), None)
    if first is None:
        return None
    out = [first if math.isnan(x) else x for x in out]
    return out


bh_nav = {}
for s in SYMBOLS:
    close = align_close(trade_dates, load_close_map(s))
    if close is None:
        continue
    bh_nav[s] = [c / close[0] for c in close]

if plt is not None:
    plt.figure(figsize=(12, 5))
    plt.plot(dates, nav, label="Strategy (MinPremium Weekly)")
    for s, n in bh_nav.items():
        plt.plot(dates, n, linewidth=1.2, alpha=0.9, label=f"Buy&Hold {s}")
    plt.title("Normalized Performance (Start = 1.0)")
    plt.xlabel("Date")
    plt.ylabel("NAV")
    plt.grid(True, alpha=0.3)
    plt.legend(ncols=2)
    plt.tight_layout()
    plt.show()


In [ ]:
# 指标 + 最近检查/交易


def daily_returns(nav_: list[float]) -> list[float]:
    out = []
    for i in range(1, len(nav_)):
        if nav_[i - 1] > 0:
            out.append(nav_[i] / nav_[i - 1] - 1.0)
    return out


def sharpe(nav_: list[float], ann_factor: int = 252) -> float:
    r = daily_returns(nav_)
    if len(r) < 2:
        return float("nan")
    mu = float(np.mean(r))
    sig = float(np.std(r, ddof=0))
    if sig <= 0:
        return 0.0
    ann_ret = (1 + mu) ** ann_factor - 1
    ann_vol = sig * math.sqrt(ann_factor)
    return float(ann_ret / ann_vol) if ann_vol > 0 else 0.0


def cagr(nav_: list[float], ann_factor: int = 252) -> float:
    r = daily_returns(nav_)
    if not r:
        return float("nan")
    return float((nav_[-1] / nav_[0]) ** (ann_factor / len(r)) - 1.0)


def max_drawdown(nav_: list[float]) -> float:
    if not nav_:
        return float("nan")
    peak = nav_[0]
    mdd = 0.0
    for x in nav_[1:]:
        if x > peak:
            peak = x
        dd = x / peak - 1.0
        if dd < mdd:
            mdd = dd
    return float(mdd)


def win_rate_daily(nav_: list[float]) -> float:
    r = daily_returns(nav_)
    if not r:
        return float("nan")
    return float(sum(1 for x in r if x > 0) / len(r))


def period_returns(dates_: list[dt.datetime], nav_: list[float], key_fn) -> list[float]:
    if len(nav_) < 2:
        return []
    groups = {}
    for d, v in zip(dates_, nav_):
        k = key_fn(d)
        if k not in groups:
            groups[k] = [v, v]
        else:
            groups[k][1] = v
    rets = []
    for first, last in groups.values():
        if first and first > 0 and last is not None:
            rets.append(last / first - 1.0)
    return rets


def period_win_rate(dates_: list[dt.datetime], nav_: list[float], key_fn) -> float:
    rets = period_returns(dates_, nav_, key_fn)
    if not rets:
        return float("nan")
    return float(sum(1 for r in rets if r > 0) / len(rets))


def summarize(name: str, dates_: list[dt.datetime], nav_: list[float]):
    return {
        "Name": name,
        "CAGR": cagr(nav_),
        "Sharpe": sharpe(nav_),
        "MaxDD": max_drawdown(nav_),
        "DailyWin": win_rate_daily(nav_),
        "MonthlyWin": period_win_rate(dates_, nav_, lambda x: (x.year, x.month)),
        "AnnualWin": period_win_rate(dates_, nav_, lambda x: x.year),
    }


rows = [summarize("Strategy", dates, nav)]
for s, n in bh_nav.items():
    rows.append(summarize(f"B&H {s}", dates, n))

for r in rows:
    print(
        f"{r['Name']:<16} CAGR={r['CAGR']:.2%} Sharpe={r['Sharpe']:.2f} MaxDD={r['MaxDD']:.2%} "
        f"DailyWin={r['DailyWin']:.2%} MonthlyWin={r['MonthlyWin']:.2%} AnnualWin={r['AnnualWin']:.2%}"
    )


hist = strat.get_check_history()
print("\nLatest weekly checks (last 10):")
for ck in hist[-10:]:
    held = ck.held_symbol_before or "<CASH>"
    imp = "-" if ck.improvement is None else f"{ck.improvement:.2f}"
    print(
        f"- {ck.trade_date} sig={ck.signal_date} {ck.week_key} held={held} best={ck.best_symbol} "
        f"held_dr={ck.held_discount_rate} best_dr={ck.best_discount_rate} imp={imp} switched={ck.switched}"
    )

if hist:
    last = hist[-1]
    held = last.held_symbol_before or "<CASH>"
    print("\nLatest decision:")
    print(" date=", last.trade_date)
    print(" held_before=", held)
    print(" best=", last.best_symbol)
    print(" improvement=", last.improvement)
    print(" switched=", last.switched)

print("\nLast trades (last 10):")
for tr in res.broker.trades[-10:]:
    print(f"- {tr.trade_date} {tr.action} {tr.symbol} exec={tr.exec_price:.4f} size={int(tr.size)} fees={tr.fees:.2f}")


print("\nDiagnostics:")
hist = strat.get_check_history()
print(" weekly checks:", len(hist))
print(" trades:", len(res.broker.trades), "(each switch ~= 2 trades; plus initial buy)")

diff = [ck for ck in hist if ck.held_symbol_before and ck.best_symbol != ck.held_symbol_before]
eligible = [ck for ck in diff if ck.improvement is not None and ck.improvement >= MIN_IMPROVEMENT]
print(" best!=held weeks:", len(diff))
print(" eligible switches (improvement>=threshold):", len(eligible))
switch_total = sum(1 for ck in hist if ck.switched)
switch_rebalance = sum(1 for ck in hist if ck.switched and ck.held_symbol_before)
print(" actual switched:", switch_total, "(incl initial)")
print(" rebalances (held->new):", switch_rebalance)

if diff:
    imps = [ck.improvement for ck in diff if ck.improvement is not None]
    if imps:
        imps_sorted = sorted(imps, reverse=True)
        p90 = imps_sorted[int(0.9 * len(imps_sorted)) - 1] if len(imps_sorted) >= 10 else None
        print(" improvement stats (pct-pt):")
        print("  max=", round(imps_sorted[0], 4), "p90=", (round(p90, 4) if p90 is not None else "-"), "median=", round(imps_sorted[len(imps_sorted)//2], 4))

    top = sorted(diff, key=lambda x: (x.improvement or float('-inf')), reverse=True)[:10]
    print(" top improvements (top 10):")
    for ck in top:
        imp = ck.improvement
        imp_str = "-" if imp is None else f"{imp:.2f}"
        print(f"  - {ck.trade_date} sig={ck.signal_date} held={ck.held_symbol_before} best={ck.best_symbol} imp={imp_str}")


def _run_case(*, min_improvement: float, monday_only: bool):
    st = ETFMinPremiumWeeklyStrategy(
        db_path_raw=str(DB_PATH),
        symbols=SYMBOLS,
        start_date=START_DATE,
        end_date=END_DATE,
        use_adjusted=USE_ADJUSTED,
        monday_only=monday_only,
        min_improvement=min_improvement,
    )
    rr = run_backtest(
        bars=bars,
        strategy=st,
        initial_cash=INIT_CASH,
        symbol="",
        enable_trade_log=False,
        mark_error_policy="warn",
    )
    return {
        "min_improvement": float(min_improvement),
        "monday_only": bool(monday_only),
        "final_nav": float(rr.final_equity) / float(INIT_CASH),
        "trades": len(rr.broker.trades),
        "checks": len(st.get_check_history()),
        "switched": sum(1 for ck in st.get_check_history() if ck.switched),
    }

print("\nSensitivity (same data window):")
for mi in [0.0, 0.5, 1.0, 2.0]:
    row = _run_case(min_improvement=mi, monday_only=True)
    print(" - weekly(Mon) thr=", mi, "final_nav=", round(row["final_nav"], 4), "trades=", row["trades"], "switched=", row["switched"])

row = _run_case(min_improvement=1.0, monday_only=False)
print(" - weekly(first trading day) thr=1.0 final_nav=", round(row["final_nav"], 4), "trades=", row["trades"], "switched=", row["switched"])
